In [35]:
# 2. Prepare dataset for GPT-2 test-drive

In [36]:
import pandas as pd

In [37]:
# Reading the data from the file
df = pd.read_csv("../data/raw/filtered.tsv", sep="\t", index_col=0)

df_rows = df.shape[0]

df.head()

,reference,translation,similarity,length_diff,ref_tox,trn_tox
0,"If Alkar is flooding her with psychic waste, t...","if Alkar floods her with her mental waste, it ...",0.785171,0.010309,0.014195,0.981983
1,Now you're getting nasty.,you're becoming disgusting.,0.749687,0.071429,0.065473,0.999039
2,"Well, we could spare your life, for one.","well, we can spare your life.",0.919051,0.268293,0.213313,0.985068
3,"Ah! Monkey, you've got to snap out of it.","monkey, you have to wake up.",0.664333,0.309524,0.053362,0.994215
4,I've got orders to put her down.,I have orders to kill her.,0.726639,0.181818,0.009402,0.999348


In [38]:
# Drop length_diff column
df = df.drop(columns=["length_diff"])

# Drop rows with similarity less than 0.5
df = df[df["similarity"] >= 0.5]

In [39]:
# Add toxicity difference column
df["tox_diff"] = df["ref_tox"] - df["trn_tox"]
df.head()

,reference,translation,similarity,ref_tox,trn_tox,tox_diff
0,"If Alkar is flooding her with psychic waste, t...","if Alkar floods her with her mental waste, it ...",0.785171,0.014195,0.981983,-0.967788
1,Now you're getting nasty.,you're becoming disgusting.,0.749687,0.065473,0.999039,-0.933567
2,"Well, we could spare your life, for one.","well, we can spare your life.",0.919051,0.213313,0.985068,-0.771755
3,"Ah! Monkey, you've got to snap out of it.","monkey, you have to wake up.",0.664333,0.053362,0.994215,-0.940853
4,I've got orders to put her down.,I have orders to kill her.,0.726639,0.009402,0.999348,-0.989946


In [40]:
# Drop rows with toxicity difference absolute value less than 0.5
df = df[df["tox_diff"].abs() >= 0.5]

# Count dropped rows
print(f"Dropped {df_rows - df.shape[0]} rows")

In [43]:
# Make a combined text column

# The format is: [s]text[/s]»[t]text[/t]
# [s] - source text
# [t] - target text
# » - separator

# If the tox_diff is negative, swap reference and translation

df["combined_text"] = df.apply(
    lambda row: f"[s]{row['reference']}[/s]»[t]{row['translation']}[/t]"
    if row["tox_diff"] > 0
    else f"[s]{row['translation']}[/s]»[t]{row['reference']}[/t]",
    axis=1,
)

df.head()

,reference,translation,similarity,ref_tox,trn_tox,tox_diff,combined_text
0,"If Alkar is flooding her with psychic waste, t...","if Alkar floods her with her mental waste, it ...",0.785171,0.014195,0.981983,-0.967788,"[s]if Alkar floods her with her mental waste, ..."
1,Now you're getting nasty.,you're becoming disgusting.,0.749687,0.065473,0.999039,-0.933567,[s]you're becoming disgusting.[/s]»[t]Now you'...
2,"Well, we could spare your life, for one.","well, we can spare your life.",0.919051,0.213313,0.985068,-0.771755,"[s]well, we can spare your life.[/s]»[t]Well, ..."
3,"Ah! Monkey, you've got to snap out of it.","monkey, you have to wake up.",0.664333,0.053362,0.994215,-0.940853,"[s]monkey, you have to wake up.[/s]»[t]Ah! Mon..."
4,I've got orders to put her down.,I have orders to kill her.,0.726639,0.009402,0.999348,-0.989946,[s]I have orders to kill her.[/s]»[t]I've got ...


In [44]:
# Save the combined column to a file without column names or index or quotes
df["combined_text"].to_csv(
    "../data/interim/combined_text.csv", index=False, header=False, quoting=3, sep="\t"
)